# Бакеты холодности: товары и юзеры вместе

Смотрим, хватит ли данных, чтобы оценивать модели отдельно по холодности товара и длине истории юзера. В `eda_compare` было только деление холодный/тёплый (5+ юзеров) по всем годам. Здесь бакетов больше и считаем их только по train, потому что на момент предсказания модель знает только прошлое. k-core не применяем: он убрал бы холодные бакеты целиком.

- граница train/test задаётся одной настройкой `CUTOFF`
- чистка как в `eda_fashion` и `eda_grocery`: `recsys.data.load.read_reviews`: убираем ошибочные строки и повторы одного отзыва (`DEDUP_KEY`)
- бакет товара: сколько разных юзеров у него в train
- бакет юзера: сколько разных товаров у него в train
- test: уникальные пары юзер-товар после `CUTOFF`, которых не было в train

In [1]:
import truststore

# до импорта s3fs и boto3, иначе ssl уходит в бесконечную рекурсию
truststore.inject_into_ssl()

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from recsys.data.load import PLACEHOLDER, read_meta, read_reviews

In [3]:
CUTOFF = pd.Timestamp("2022-01-01")  # train: раньше этой даты, test: с этой даты
CATEGORIES = ["Amazon_Fashion", "Grocery_and_Gourmet_Food"]

ITEM_BINS = [-1, 0, 1, 4, 19, float("inf")]
ITEM_LABELS = ["0", "1", "2-4", "5-19", "20+"]
USER_BINS = [-1, 0, 1, 4, float("inf")]
USER_LABELS = ["0", "1", "2-4", "5+"]

# меньше этого числа пар в клетке считаем слишком мало для отдельной оценки
MIN_TEST_PAIRS = 1000

## Загрузка и чистка

In [4]:
def load_reviews(category):
    columns = ["user_id", "parent_asin", "timestamp"]
    reviews = read_reviews(category, columns=columns)
    print(f"{category}: отзывов после чистки {len(reviews):,}")
    return reviews


def load_catalog(category):
    columns = [
        "parent_asin",
        "title",
        "description",
        "features",
        "images",
        "rating_number",
    ]
    meta = read_meta(category, columns=columns)
    desc = meta["description"].str.join(" ").fillna("").str.strip() != ""
    feat = meta["features"].str.join(" ").fillna("").str.strip() != ""
    placeholder = meta["images"].str.contains(PLACEHOLDER, regex=False)
    return pd.DataFrame(
        {
            "parent_asin": meta["parent_asin"],
            "has_title": meta["title"].notna() & (meta["title"].str.strip() != ""),
            "has_description": desc,
            "has_features": feat,
            # текст кроме title: description или features
            "has_extra_text": desc | feat,
            "has_image": (meta["images"] != "[]") & ~placeholder,
            # число оценок на Amazon на дату сбора, включая оценки без текста
            "rating_number": meta["rating_number"].fillna(0).astype(int),
        }
    )

## Бакеты

In [5]:
def bucket(counts, bins, labels):
    return pd.cut(counts, bins=bins, labels=labels)


def split_pairs(reviews, cutoff):
    train = reviews[reviews["timestamp"] < cutoff]
    test = reviews[reviews["timestamp"] >= cutoff]
    pair = ["user_id", "parent_asin"]
    train_pairs = train[pair].drop_duplicates()
    test_pairs = test[pair].drop_duplicates()
    merged = test_pairs.merge(train_pairs, on=pair, how="left", indicator=True)
    new_pairs = merged.loc[merged["_merge"] == "left_only", pair]
    print(
        f"train пар {len(train_pairs):,}, test пар {len(test_pairs):,}, "
        f"из них новых {len(new_pairs):,}"
    )
    return train_pairs, new_pairs.reset_index(drop=True)


def share(series):
    return (100 * series / series.sum()).round(1)


def analyze(category, cutoff=CUTOFF):
    reviews = load_reviews(category)
    catalog = load_catalog(category)
    train, test = split_pairs(reviews, cutoff)

    item_n = train.groupby("parent_asin").size()
    user_n = train.groupby("user_id").size()

    catalog["n_users"] = catalog["parent_asin"].map(item_n).fillna(0).astype(int)
    catalog["item_bucket"] = bucket(catalog["n_users"], ITEM_BINS, ITEM_LABELS)
    test = test.assign(
        n_users=test["parent_asin"].map(item_n).fillna(0).astype(int),
        n_items=test["user_id"].map(user_n).fillna(0).astype(int),
    )
    test["item_bucket"] = bucket(test["n_users"], ITEM_BINS, ITEM_LABELS)
    test["user_bucket"] = bucket(test["n_items"], USER_BINS, USER_LABELS)

    in_catalog = test["parent_asin"].isin(catalog["parent_asin"]).mean()
    print(f"товары из test есть в каталоге: {in_catalog:.2%}")

    items = pd.DataFrame(
        {
            "товаров": catalog.groupby("item_bucket", observed=False).size(),
            "товаров в test": test.groupby("item_bucket", observed=False)[
                "parent_asin"
            ].nunique(),
            "пар в test": test.groupby("item_bucket", observed=False).size(),
        }
    )
    items.insert(1, "% каталога", share(items["товаров"]))
    items["% пар"] = share(items["пар в test"])

    users = pd.DataFrame(
        {
            "юзеров в test": test.groupby("user_bucket", observed=False)[
                "user_id"
            ].nunique(),
            "пар в test": test.groupby("user_bucket", observed=False).size(),
        }
    )
    users.insert(1, "% юзеров", share(users["юзеров в test"]))
    users["% пар"] = share(users["пар в test"])

    by = ["user_bucket", "item_bucket"]
    joint = (
        test.groupby(by, observed=False)
        .agg(
            pairs=("user_id", "size"),
            users=("user_id", "nunique"),
            items=("parent_asin", "nunique"),
        )
        .reset_index()
    )
    joint["% пар"] = (100 * joint["pairs"] / joint["pairs"].sum()).round(2)
    joint["достаточно"] = joint["pairs"] >= MIN_TEST_PAIRS

    flags = [
        "has_title",
        "has_description",
        "has_features",
        "has_extra_text",
        "has_image",
    ]
    content = (
        catalog.groupby("item_bucket", observed=False)[flags].mean() * 100
    ).round(1)
    in_test = catalog[catalog["parent_asin"].isin(test["parent_asin"])]
    content_test = (
        in_test.groupby("item_bucket", observed=False)[flags].mean() * 100
    ).round(1)

    return {
        "reviews": reviews,
        "catalog": catalog,
        "items": items,
        "users": users,
        "joint": joint,
        "content": content,
        "content_test": content_test,
    }


results = {category: analyze(category) for category in CATEGORIES}

Amazon_Fashion: отзывов после чистки 2,475,694


train пар 2,266,742, test пар 207,683, из них новых 207,633


товары из test есть в каталоге: 100.00%


Grocery_and_Gourmet_Food: отзывов после чистки 14,187,557


train пар 11,203,094, test пар 2,885,200, из них новых 2,878,074


товары из test есть в каталоге: 100.00%


## Сверка с `eda_compare`

Если считать по всем годам, доля товаров с 5+ юзерами должна совпасть с тёплыми из прошлых ноутбуков: 11.3% у Fashion и 42.9% у Grocery.

In [6]:
for category, res in results.items():
    pairs = res["reviews"][["user_id", "parent_asin"]].drop_duplicates()
    n_users = res["catalog"]["parent_asin"].map(pairs.groupby("parent_asin").size())
    warm = n_users.fillna(0) >= 5
    print(f"{category}: тёплых за все годы {warm.sum():,} ({warm.mean():.1%})")

Amazon_Fashion: тёплых за все годы 93,728 (11.3%)


Grocery_and_Gourmet_Food: тёплых за все годы 259,082 (42.9%)


## Товары

In [7]:
for category, res in results.items():
    print(category)
    display(res["items"])

Amazon_Fashion


,товаров,% каталога,товаров в test,пар в test,% пар
item_bucket,,,,,
0,75156,9.1,74917,115307,55.5
1,444922,53.9,10025,15069,7.3
2-4,217674,26.3,8262,15943,7.7
5-19,76291,9.2,5818,19482,9.4
20+,12065,1.5,2587,41832,20.1


Grocery_and_Gourmet_Food


,товаров,% каталога,товаров в test,пар в test,% пар
item_bucket,,,,,
0,79077,13.1,78985,360209,12.5
1,158060,26.2,20356,59359,2.1
2-4,145530,24.1,33920,121085,4.2
5-19,135614,22.5,58134,337872,11.7
20+,84993,14.1,60876,1999549,69.5


## Юзеры

In [8]:
for category, res in results.items():
    print(category)
    display(res["users"])

Amazon_Fashion


,юзеров в test,% юзеров,пар в test,% пар
user_bucket,,,,
0,165269,85.7,174949,84.3
1,18746,9.7,20690,10.0
2-4,7354,3.8,8701,4.2
5+,1588,0.8,3293,1.6


Grocery_and_Gourmet_Food


,юзеров в test,% юзеров,пар в test,% пар
user_bucket,,,,
0,1248737,66.2,1645195,57.2
1,278432,14.8,409177,14.2
2-4,232083,12.3,408979,14.2
5+,126089,6.7,414723,14.4


## Товары и юзеры вместе

Строки это бакет юзера, столбцы это бакет товара, в клетках число пар в test.

In [9]:
for category, res in results.items():
    joint = res["joint"]
    print(category)
    for value in ["pairs", "% пар", "users", "items"]:
        print(value)
        display(joint.pivot(index="user_bucket", columns="item_bucket", values=value))
    print(
        "клеток с достаточным числом пар:", joint["достаточно"].sum(), "из", len(joint)
    )

Amazon_Fashion
pairs


item_bucket,0,1,2-4,5-19,20+
user_bucket,,,,,
0,94374,12600,13668,16998,37309
1,12535,1583,1496,1768,3308
2-4,5758,686,616,562,1079
5+,2640,200,163,154,136


% пар


item_bucket,0,1,2-4,5-19,20+
user_bucket,,,,,
0,45.45,6.07,6.58,8.19,17.97
1,6.04,0.76,0.72,0.85,1.59
2-4,2.77,0.33,0.30,0.27,0.52
5+,1.27,0.10,0.08,0.07,0.07


users


item_bucket,0,1,2-4,5-19,20+
user_bucket,,,,,
0,87982,12551,13608,16917,37116
1,11268,1564,1485,1750,3281
2-4,4863,668,604,555,1065
5+,1282,178,150,138,133


... (7 outputs left out)

item_bucket,0,1,2-4,5-19,20+
user_bucket,,,,,
0,169303,31638,63788,168324,937375
1,37381,8265,16658,43960,212490
2-4,33761,8266,16399,43091,183039
5+,26324,7883,15209,35341,105962


items


item_bucket,0,1,2-4,5-19,20+
user_bucket,,,,,
0,58704,14145,24690,46485,56429
1,24648,5799,11046,25009,42782
2-4,23781,5801,11186,25624,42878
5+,22552,5618,11071,25742,41736


клеток с достаточным числом пар: 20 из 20


## Контент по бакетам

Доля товаров с контентом, в %. Первая таблица по всему каталогу, вторая только по товарам, которые встречаются в test.

In [10]:
for category, res in results.items():
    print(category, "весь каталог")
    display(res["content"])
    print(category, "товары из test")
    display(res["content_test"])

Amazon_Fashion весь каталог


,has_title,has_description,has_features,has_extra_text,has_image
item_bucket,,,,,
0,100.0,9.1,68.6,69.4,97.4
1,100.0,6.7,51.8,52.7,98.4
2-4,100.0,6.6,57.4,58.0,98.9
5-19,100.0,8.5,62.6,63.2,99.4
20+,100.0,15.6,67.4,68.3,99.5


Amazon_Fashion товары из test


,has_title,has_description,has_features,has_extra_text,has_image
item_bucket,,,,,
0,100.0,9.2,68.7,69.5,97.4
1,100.0,11.7,71.0,71.8,98.6
2-4,100.0,16.4,72.0,73.0,98.7
5-19,100.0,25.3,76.1,77.0,99.4
20+,100.0,32.7,84.5,85.4,99.7


Grocery_and_Gourmet_Food весь каталог


,has_title,has_description,has_features,has_extra_text,has_image
item_bucket,,,,,
0,100.0,58.0,79.4,84.9,99.7
1,100.0,60.5,63.8,71.7,99.3
2-4,100.0,62.0,67.8,74.7,99.7
5-19,100.0,64.8,74.6,80.2,99.9
20+,100.0,68.7,85.0,88.8,99.9


Grocery_and_Gourmet_Food товары из test


,has_title,has_description,has_features,has_extra_text,has_image
item_bucket,,,,,
0,100.0,58.0,79.4,84.9,99.7
1,100.0,62.6,73.6,80.9,99.9
2-4,100.0,63.3,75.6,82.0,99.9
5-19,100.0,65.2,79.7,85.3,99.9
20+,100.0,67.8,87.2,91.0,99.9


## Покрытие по популярности на Amazon

В датасете не все оценки товара с Amazon, а только отзывы с текстом, которые собрали авторы. Сравниваем число отзывов товара в данных (после чистки, за все годы) с `rating_number`, числом оценок на Amazon на дату сбора. Бакеты здесь по `rating_number`, а не по истории в train.

Покрытие больше 100% в бакете `1` не ошибка чистки. У 0.18% таких товаров Fashion и 0.53% Grocery в данных 2+ отзыва от разных юзеров, часто за несколько лет и по нескольким `asin`. `rating_number` и отзывы сняты в разное время, а вариации товара могли перегруппировать или счётчик сбросить. Таких товаров 0.1% у Fashion и 0.4% у Grocery по всему каталогу, на остальные бакеты это почти не влияет. Бакет `1` читаем как «покрыт полностью».


In [11]:
RATING_BINS = [0, 1, 9, 99, 999, float("inf")]
RATING_LABELS = ["1", "2-9", "10-99", "100-999", "1000+"]


def coverage(res):
    n_reviews = res["reviews"].groupby("parent_asin").size()
    catalog = res["catalog"]
    catalog = catalog[catalog["rating_number"] > 0].assign(
        n_reviews=lambda d: d["parent_asin"].map(n_reviews).fillna(0).astype(int),
        popularity=lambda d: bucket(d["rating_number"], RATING_BINS, RATING_LABELS),
    )
    by = catalog.groupby("popularity", observed=False)
    table = pd.DataFrame(
        {
            "товаров": by.size(),
            "оценок на Amazon": by["rating_number"].sum(),
            "отзывов в данных": by["n_reviews"].sum(),
        }
    )
    table["% товаров"] = share(table["товаров"])
    table["% оценок на Amazon"] = share(table["оценок на Amazon"])
    table["% отзывов в данных"] = share(table["отзывов в данных"])
    ratio = table["отзывов в данных"] / table["оценок на Amazon"]
    table["покрытие, %"] = (100 * ratio).round(1)
    total = catalog["n_reviews"].sum() / catalog["rating_number"].sum()
    print(f"всего покрытие {total:.1%}")
    return table


for category, res in results.items():
    print(category)
    res["coverage"] = coverage(res)
    display(res["coverage"])

Amazon_Fashion


всего покрытие 16.7%


,товаров,оценок на Amazon,отзывов в данных,% товаров,% оценок на Amazon,% отзывов в данных,"покрытие, %"
popularity,,,,,,,
1,175277,175277,176610,21.2,1.2,7.1,100.8
2-9,443883,1881868,747872,53.7,12.7,30.2,39.7
10-99,190465,4710979,1029775,23.1,31.8,41.6,21.9
100-999,15102,3763540,361200,1.8,25.4,14.6,9.6
1000+,1381,4290534,160237,0.2,28.9,6.5,3.7


Grocery_and_Gourmet_Food


всего покрытие 10.9%


,товаров,оценок на Amazon,отзывов в данных,% товаров,% оценок на Amazon,% отзывов в данных,"покрытие, %"
popularity,,,,,,,
1,50766,50766,55608,8.4,0.0,0.4,109.5
2-9,184251,855650,417996,30.5,0.7,2.9,48.9
10-99,240550,8410540,2165256,39.9,6.4,15.3,25.7
100-999,103604,32698595,4734723,17.2,25.0,33.4,14.5
1000+,24103,88629277,6813974,4.0,67.8,48.0,7.7


## Сохраняем таблицы

In [12]:
out = Path("../reports")
stamp = CUTOFF.date().isoformat()
for name in ["items", "users", "joint", "content", "content_test", "coverage"]:
    frames = [
        results[category][name].assign(category=category, cutoff=stamp)
        for category in CATEGORIES
    ]
    pd.concat(frames).to_csv(out / f"eda_coldness_{name}.csv", index=name != "joint")

## Итог

**Хватает ли данных.** У Grocery во всех 20 клетках больше 1000 пар в test, самая маленькая 8 556. У Fashion только в 13 из 20: юзеры с 5+ товарами и товары с 1-19 юзерами почти пустые. Для Fashion бакеты юзеров стоит объединить в 0, 1 и 2+, но и тогда часть клеток будет близка к порогу.

**Куда попадает test.** В основном к юзерам без истории в train: 84.3% пар у Fashion и 57.2% у Grocery. У Fashion 55.5% пар приходится на товары без истории, 45.45% это и юзер, и товар без истории. У Grocery наоборот, 69.5% пар на товары с 20+ юзерами.

**k-core.** Пары, где и у юзера, и у товара 5+ взаимодействий в train, это 0.14% test у Fashion и 11.3% у Grocery. Если оценивать только на них, почти весь test потеряется.

**Контент.** Хуже всего заполнен бакет 1, а не 0: новые товары описаны лучше старых редких. features у Fashion 51.8% в бакете 1, 68.6% в бакете 0, 67.4% в 20+. У Grocery 63.8%, 79.4% и 85.0%. Если сравнивать бакет 1 и 20+, разница больше 10 п.п. в обеих категориях, поэтому вывод `eda_fashion` про маленькую разницу у Fashion при делении на большее число бакетов не держится. Картинки есть у 97.4% и больше товаров в каждом бакете, title почти у всех.

**Ограничение.** Бакеты зафиксированы на дате `CUTOFF`: покупки внутри test в историю не добавляются. Если в протоколе оценки история будет обновляться, доля юзеров без истории станет меньше.

**Выборка неравномерна по популярности.** В датасете только отзывы с текстом, а `rating_number` на Amazon считает все оценки, в том числе без текста. Отзывов в данных 16.7% от `rating_number` у Fashion и 10.9% у Grocery, и чем популярнее товар, тем меньше доля: у товаров с 1000+ оценок 3.7% и 7.7%, у товаров с 2-9 оценками 39.7% и 48.9%. Популярные товары в данных представлены слабее, чем на самом Amazon. Поэтому выводы проекта (насколько силён бейзлайн по популярности, как холодные товары соотносятся с тёплыми) могут не перенестись на настоящую систему Amazon. `rating_number` как признак не используем: он снят на дату сбора и включает оценки после `CUTOFF`.